### **ACID - PART 4b - Segment object**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2026/03/30

## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [1]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
import napari
from skimage.transform import resize
# import tifffile
# import matplotlib.pyplot as plt
from acid.utils.listdirNHF import listdirNHF
from acid.utils.get_defaults import default_file_name, default_multifile_name
from acid.utils.fov_axis_utils import get_fov_ch_shape
from acid.utils.save_image import tifffile_save_ometiff
from acid.image_processing.extract_metadata import extract_ometif_imagej_metadata
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modified are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# indicate the path to the directory storing the images to segment
# NOTE: this are expected to be the fields of view saved after background
# correction (part4b notebook)
proc_fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov_proc"

# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from either part4a or part3 notebook
# metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"
metadata_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\proc_metadata"

# # indicate the path to the directory where outputs will be saved
# NOTE: if the directory does not exist, the pipeline will try to create it
# output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov_proc"
output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\seg"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part4b notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "default"



# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# # --- parameters to select the train set ---
# indicate the name of the column indicating whether the row belongs to train or test set
is_train_column="is_train"

# indicate the value signalling that a row (aka a field of view) belongs to the train set
train_val=1



# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv" # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_metadata_file_exclude = None # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
# if metadata_from_file_name is True
metadata_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if metadata_from_file_name is True
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if metadata_from_file_name is True
metadata_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True


# --- parameters for saving metadata within the saved segmentation mask ---
# image segmentation - name of processing date in metadata - this is the name
# of the entry in the metadata dictionary to save within the segmentation mask.
# The entry indicates the date when the segmentation was done.
proc_img_meta_date_name = "processing_date_yymmdd"

# image segmentation - date format in metadata - this is the format to use for indicating
# the date when the segmentation was done in the metadata saved within the segmentation mask
processing_date_format = '%y%m%d'

# image segmentation - data type in metadata - this is the entry to use for indicating
# the data type of the segmentation mask in the metadata saved within the segmentation mask
proc_img_meta_dtype_name = 'dtype'


# indicate the photometric interpretation to be used when saving the ome.tif files
# NOTE: at the moment, only 'minisblack' has been tested
photometric = 'minisblack'

# indicate whether to save the background function image with ImageJ compatible
save_imagej_compatible = True



# --- parameters for metadata dataframe updating ---
# column name word separator - this is the separator to use for separating the different parts of
# the column names to be added to the metadata dataframe
column_name_separator = "_"

# # channel name separator - this is the separator to use for separating the channel name from the rest of
# # the column name to be added to the metadata dataframe.
# ch_name_separator = "-"

# image segmentation metadata dataframe - computation date column name - this is the name of the column
# to be added to the metadata dataframe to indicate the day when the image segmentation was done.
seg_df_date_clm_name = f"segmentation{column_name_separator}date"

# image segmentation metadata dataframe - computation date format - this is the format to be used for
# indicating the date when the image segmentation was done. This is used for saving the date in the
# metadata dataframe
seg_df_meta_date_format = '%y%m%d'

# image segmentation metadata dataframe - segmentation mask column name - this is the name of the column
# to be added to the metadata dataframe to indicate the name used for saving the segmentation file.
seg_df_file_name_clm_name = f"segmentation{column_name_separator}file{column_name_separator}name"
















# --- parameters for image segmentation ---
# image segmentation - corrected file column name - this is the name of the column
# with the name used for saving the illumination corrected file.
illum_correct_df_file_name_clm_name = f"illumination{column_name_separator}correction{column_name_separator}file{column_name_separator}name"

# what is expected to be and what will be used as null value
null_value = np.nan
















# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# include indexes when saving pandas dataframes as csv files
save_csv_index = False # if False, the index will not be saved as a separate column in the csv file

# segmentation mask name - savingword - this is the word to use in the segmentation mask file name
# to indicate that the file is a segmentation mask.
segmentation_mask_savingword = 'seg'

# ome suffix - used to save ome.tif files
ome_suffix = ".ome.tif"

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}5.csv"

# processing metadata dataframe name - date format
metadata_date_format = '%Y%m%d'

# hyperparameters dataframe name - date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters dataframe name - savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters dataframe name - file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}5.csv"



# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error



### Create output directory and secondary output directory if they don't exist

##### Output directory stores the segmentation masks
##### Secondary output directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [4]:
# create the output_directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=exist_ok)

# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)


#### Open the metadata dataframe - this is expected to be the output of part 4b

Run the following cell.

Don't modify the following cell.

In [5]:
# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)
    
    # get the default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           from_file_name=metadata_from_file_name,
                                           directory_path=metadata_directory,
                                           separator=metadata_default_separator,
                                           date_position=metadata_default_date_position,
                                           date_format=metadata_default_date_format,
                                           reverse=metadata_default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df


using 20251205_ACID_metadata_part_2.csv as default metadata file


#### Select train data set - NOTE: the metadata_df is updated into a metadata_df which does not contain the test data

Run the following cell.

Don't modify the following cell.

In [6]:
# Select only the train set and update metadata_df
metadata_df = metadata_df[metadata_df[is_train_column] == train_val]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong to the train set."
assert all(metadata_df[is_train_column] == train_val), "Not all rows in metadata_df belong to the train set."

# Display the metadata dataframe
metadata_df

,Unnamed: 0,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,...,size_c,size_z,size_y,physical_size_y,size_x,physical_size_x,dims_order,int_well,treatment,is_train
1,1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
2,2,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A3,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
3,3,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A4,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
5,5,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
6,6,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A7,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,92,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G2,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,5,infected_nocpd,1
93,93,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G3,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,5,infected_nocpd,1
95,95,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G5,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,5,infected_nocpd,1
96,96,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G6,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,5,1,1024,0.325,1024,0.325,CYX,5,infected_nocpd,1


### Import CellPose and CellPose models - NOTE: this is kept separated from previous imports to allow notebook modularity

Run the following cell.

Don't modify the following cell.

In [7]:
# Import required modules
from cellpose import models
from cellpose.io import imread
import torch

# Use gpu if available else cpu
if torch.cuda.is_available():
    use_gpu = True
    print("---------")
    print("GPU is available")
else:
    use_gpu = False
    print("---------")
    print("GPU is not available")

# Import segmentation model
model = models.CellposeModel(gpu=use_gpu)


---------
GPU is not available


100%|██████████| 1.15G/1.15G [00:21<00:00, 57.1MB/s]


### MAIN LOOP
#### 5.1. Preprocess field of views.
#### 5.2. Segment cell and nucleus.
#### 5.2. Save segmentation masks
#### 5.3. Update and save metadata

## **--- --- ---**

Run the following cell.

Don't modify the following cell.

In [ ]:

# seg_mentation_metadata_dict = {f'segmentation_date_yymmdd{nucleus_suffix}{cell_suffix}':[], #to check
#                              'cell_segmentation':[], #to check
#                              'nucleus_segmentation':[]} #to check

# initialize lists to collect processing metadata for updating - this will be used to update the metadata df
segmentation_date_collection = []
segmentation_file_name_collection = []


# iterate through the rows of the metadata dataframe
for file_idx in metadata_df.index[:3]:
    print("---------")

    # ---------   ---------
    # OPEN THE FIELD OF VIEW FILE
    # ---------   ---------
    try:
        # get the name of the field of view as ome.tif file
        field_of_view_file = metadata_df.loc[file_idx, illum_correct_df_file_name_clm_name]

        img = imread(os.path.join(proc_fov_directory,field_of_view_file))

        print(f"working on {field_of_view_file}")
    except:
        print(f"can't open {field_of_view_file}, skipping image segmentation")

        segmentation_date_collection.append(null_value)
        segmentation_file_name_collection.append(null_value)
        continue

    # ---------   ---------
    # PREPROCESS IMAGE
    # Select nucleus and actin channels
    # filter nucleus using a 10x10 median filtering
    # stack non-filtered nucleus and actin-channel into a new array and filter the channels, individually, using a 3x3 median filtering
    # 2x2 binning
    # NOTE: no dtype conversion is needed as resizing automatically changes image to float
    # ---------   ---------

    # Select nucleus and actin channels
    unstacked_img = np.unstack(img, axis=channel_axis)
    nucleus_channel = unstacked_img[nucleus_position]
    actin_channel = unstacked_img[actin_position]
    restacked_img = np.stack([nucleus_channel, actin_channel], axis=channel_axis)

    # Preprocess nucleus and nucleus-actin-stack
    preproc_nucleus = preprocess_channel(image=nucleus_channel,
                                             med_size=med_size_nucleus,
                                             factor=factor)

    preproc_img = preprocess_fov(image=actin_channel,
                                       channel_axis=channel_axis,
                                       med_size=med_size_cell,
                                       factor=factor)
        
    print("Image preprocessing is done")

    # ---------   ---------
    # SEGMENT CELLS
    # ---------   ---------

    # calculate the original size of the 2D image (aka, the size of each imaged channel)
    # this will be used to resize the segmented masks to their original size, as masks are calculated on binned images
    original_ch_img_size = tuple([s for p,s in enumerate(img.shape) if p!=channel_axis])

    print("Cellular segmentation is beginning. Please wait...")
    cell_masks_i, cell_flows, cell_styles = model.eval(preproc_img, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold, diameter=diameter)
        
    # re-expand segmentation mask
    cell_masks = resize(cell_masks_i, output_shape=original_ch_img_size, order=resize_order)

    print("Cellular segmentation is done")

    # ---------   ---------
    # SEGMENT NUCLEI
    # NOTE: no filter diameter is used, small structures are useful to be detected, so that they are subtracted from cytosol masks
    # ---------   ---------
    print("Nuclear segmentation is beginning. Please wait...")
    nuc_masks_i, nuc_flows, nuc_styles = model.eval(preproc_nucleus, flow_threshold=0.9, cellprob_threshold=0.5) #to check
        
    # re-expand segmentation mask
    nuc_masks = resize(nuc_masks_i, output_shape=original_ch_img_size, order=resize_order) #to check
        
    print("Nuclear segmentation is done")

    # # ---------   ---------
    # # SEGMENT CYTOSOLS - THIS HAS BEEN MOVED TO PART 2
    # # ---------   ---------
    # print("Cytoplasm segmentation is beginning. Please wait...") #to check
    # cyt_masks = np.where(nuc_masks>0,0,cell_masks).astype(np.uint16) #to check
    # print("Cytoplasm segmentation is done") #to check

    # ---------   ---------
    # SAVE RESULTS
    # ---------   ---------
        
    # form saving names
    cell_save_name = f"{fov_ome_name.removesuffix(ome_suffix)}{cell_suffix}{ome_suffix}"

    nuc_save_name = f"{fov_ome_name.removesuffix(ome_suffix)}{nucleus_suffix}{ome_suffix}"
        
    # cyt_save_name = f"{fov_ome_name.removesuffix(ome_suffix)}{cytosol_suffix}{ome_suffix}" #to check

    # form a metadata dictionary to be used for saving metadata in the segmentation masks
    segmentation_metadata_dict = {'processing_date_yymmdd':datetime.datetime.now().strftime('%y%m%d')}
    for clm in metadata_df.columns:
        if clm in ['raw_file_name', 'scene_name', 'ome_tif_file_name', 'location', 'microscope', 'objective', 'donor',
                       'transfection', 'stiffness', 'stimulation', 'time_of_stimulation',
                       'physical_size_unit_x', 'physical_size_unit_y', 'physical_size_y', 'size_x',
                       'physical_size_x']:
                
            segmentation_metadata_dict[clm]=metadata_df.loc[file_idx,clm]
        
    # change metadata dictionary to an ImageJ comaptible format
    imagej_segmentation_metadata_dict = imagej_compatible_metadata_dict(segmentation_metadata_dict)

    # save file
    tifffile_save_ometiff(os.path.join(segmentation_directory,cell_save_name),
                                    data=cell_masks,
                                    imagej=True,
                                    photometric="minisblack",
                                    metadata=imagej_segmentation_metadata_dict)

    tifffile_save_ometiff(os.path.join(segmentation_directory,nuc_save_name),
                                    data=nuc_masks,
                                    imagej=True,
                                    photometric="minisblack",
                                    metadata=imagej_segmentation_metadata_dict)
        
    
    print("Segmentation results have been saved")

    # ---------   ---------
    # COLLECT FILE SAVE NAME AND SEGMENTATION DATE TO UPDATE METADATA DATAFRAME
    # ---------   ---------
    # seg_mentation_metadata_dict['segmentation_date_yymmdd'].append(datetime.datetime.now().strftime('%y%m%d')) #to check
    seg_mentation_metadata_dict[f'segmentation_date_yymmdd{nucleus_suffix}{cell_suffix}'].append(datetime.datetime.now().strftime('%y%m%d')) #to check
    seg_mentation_metadata_dict['cell_segmentation'].append(cell_save_name)
    seg_mentation_metadata_dict['nucleus_segmentation'].append(nuc_save_name)
    # seg_mentation_metadata_dict['cytosol_segmentation'].append(cyt_save_name) #to check

# update metadata_update_dict with nan values if the file was not processed
else:
    # seg_mentation_metadata_dict['segmentation_date_yymmdd'].append(np.nan) #to check
    seg_mentation_metadata_dict[f'segmentation_date_yymmdd{nucleus_suffix}{cell_suffix}'].append(np.nan) #to check
    seg_mentation_metadata_dict['cell_segmentation'].append(np.nan)
    seg_mentation_metadata_dict['nucleus_segmentation'].append(np.nan)
    # seg_mentation_metadata_dict['cytosol_segmentation'].append(np.nan) #to check

print("")
print("Segmentation completed")

# ---------   ---------
# UDDATE METADATA DICTIONARY
# ---------   ---------

# use seg_mentation_metadata_dict to for a dataframe
segmentation_metadata_df = pd.DataFrame.from_dict(seg_mentation_metadata_dict)

# add segmentation information to the metadata dataframe
glob_metadata_df = pd.concat([metadata_df,segmentation_metadata_df], axis=1)


# ---------   ---------
# SAVE THE FINAL METADATA FILE
# ---------   ---------
glob_metadata_df.to_csv(os.path.join(proc_file_info_metadata_directory, f"{datetime.datetime.now().strftime('%y%m%d')}_nef_translocation_metadata.csv"))

print("")
print("Processing metadata saved")



---------
working on D26_GFP_Glass_CD3CD28_15min__s0.ome.tif
uint16
Image preprocessing is done
Cellular segmentation is beginning. Please wait...
(2720, 1360)
Cellular segmentation is done
Nuclear segmentation is beginning. Please wait...
Nuclear segmentation is done
Segmentation results have been saved
---------
working on D26_GFP_Glass_CD3CD28_15min__s1.ome.tif
uint16
Image preprocessing is done
Cellular segmentation is beginning. Please wait...
(2720, 1360)
Cellular segmentation is done
Nuclear segmentation is beginning. Please wait...
Nuclear segmentation is done
Segmentation results have been saved
---------
working on D26_GFP_Glass_CD3CD28_15min__s2.ome.tif
uint16
Image preprocessing is done
Cellular segmentation is beginning. Please wait...
(2720, 1360)
Cellular segmentation is done
Nuclear segmentation is beginning. Please wait...
Nuclear segmentation is done
Segmentation results have been saved

Segmentation completed

Processing metadata saved


### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# collect hyperparameters in a dictionary

hyperparameter_dict = {'input_directory':input_directory,
                       'output_directory':output_directory,
                       'channel_0':channel_0,
                       'channel_1':channel_1,
                       'channel_2':channel_2,
                       'channel_3':channel_3,
                       'channel_4':channel_4,
                       'size_unit':size_unit,
                       'file_name_separator':file_name_separator,
                       'location':location,
                       'microscope':microscope,
                       'objective':objective,
                       'nucleus_position':nucleus_position,
                       'actin_position':actin_position,
                       'input_file_target':input_file_target,
                       'input_file_exclude':input_file_exclude,
                       'ome_suffix':ome_suffix,
                       'med_size_nucleus':med_size_nucleus,
                       'med_size_cell':med_size_cell,
                       'channel_axis':channel_axis,
                       'dims_order_name':dims_order_name,
                       'factor':factor,
                       'use_gpu':use_gpu,
                       'diameter':diameter,
                       'flow_threshold':flow_threshold,
                       'cellprob_threshold':cellprob_threshold,
                       'resize_order':resize_order,
                       'cell_suffix':cell_suffix,
                       'nucleus_suffix':nucleus_suffix}


# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}_nef_translocation_hyperparameters_part1.csv"
hyperparameter_series.to_csv(os.path.join(secondary_output_path,hyperparameter_saving_name))
